# Proyecto Final · Limpieza de datos (1/2)
## Abandono escolar en México: factores socioeconómicos y razones por sexo

**Pregunta de investigación (núm. 12 del pool):** ¿Qué factores socioeconómicos (ingreso del hogar, escolaridad de los padres, zona) se asocian con el abandono escolar, y cuáles son las razones más frecuentes reportadas por sexo del estudiante?

**Autor:** Manuel Aguirre · Licenciatura en Finanzas Internacionales, UDEM

**Fuente:** INEGI, *Encuesta Nacional sobre Acceso y Permanencia en la Educación (ENAPE) 2021*, datos abiertos oficiales (`enape_2021_bd_csv.zip`).

| Característica | Detalle |
|---|---|
| Levantamiento | 16 nov – 17 dic 2021, entrevista **telefónica** |
| Ciclos que cubre | 2020-2021 y 2021-2022 (en plena pandemia de COVID-19) |
| Tablas | `TVIVIENDA` (22,719 viviendas, 17 variables) y `TMODULO` (32,343 personas de 0 a 29 años, 101 variables) |
| Población que representa | ≈ 60 millones de personas de 0 a 29 años (variable `FACTOR`) |

**Flujo del notebook:** carga → diagnóstico de calidad → selección de variables → clasificación (catálogos) → limpieza con Data Wrangler → variables derivadas → validaciones → exportación (CSV + Excel).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

CRUDOS = Path("datos_crudos")
LIMPIOS = Path("datos_limpios")
LIMPIOS.mkdir(exist_ok=True)

## 1. Carga de la base de datos oficial

Los dos CSV vienen del zip oficial de INEGI y se guardan sin modificar en `datos_crudos/`.
Se leen **como texto** (`dtype=str`) porque INEGI codifica las respuestas con ceros a la izquierda (`"01"`, `"02"`…). Si se leyeran como número, `"01"` y `"1"` se confundirían y se perderían los folios con ceros iniciales.

In [2]:
viv = pd.read_csv(CRUDOS / "TVIVIENDA.csv", dtype=str)
mod = pd.read_csv(CRUDOS / "TMODULO.csv", dtype=str)
print("TVIVIENDA:", viv.shape, "| TMODULO:", mod.shape)
mod.head()

TVIVIENDA: (22719, 17) | TMODULO: (32343, 101)


,FOLIO,N_REN,SEXO,EDAD,P3_1,P3_2,PA3_1,PA3_2,PA3_3_NIVEL,PA3_3_BIMESTRE,PA3_3_TRIMESTRE,PA3_3_CUATRIMESTRE,PA3_3_SEMESTRE,PA3_3_ANIO,PA3_3_MODMAT,PA3_4,PA3_5,PA3_6,PA3_7_1,PA3_7_2,PA3_7_3,PA3_8_1,PA3_8_2,PA3_8_3,PA3_8_4,PA3_8_5,PA3_8_6,PA3_8_7,PA3_8_8,PB3_1,...,PB3_13_6,PB3_13_7,PB3_14,PB3_15,PB3_16_1,PB3_16_2,PB3_16_3,PB3_16_4,PB3_16_5,FILTRO_C,PC3_1,PC3_2,PC3_3_1,PC3_3_2,PC3_4,PC3_5,PC3_6,PC3_7,PC3_8,FILTRO_D,PD3_1,PD3_2,PD3_3,ENT,NIVEL_A,GRADO_A,NIVEL_B,GRADO_B,ESC,FACTOR
0,00000002,01,1,13,02,NaN,1,1,03,NaN,NaN,NaN,NaN,06,NaN,1,NaN,3,2,NaN,NaN,1,1,2,2,1,2,2,2,1,...,2,2,00,7,1,2,2,2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN,24,03,6,03,1,2,809
1,00000002,02,2,19,02,NaN,1,1,09,NaN,NaN,NaN,NaN,01,NaN,1,NaN,3,1,2,2,1,1,2,2,1,2,2,2,1,...,NaN,NaN,NaN,NaN,1,2,2,2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,7,NaN,NaN,24,09,1,08,2,5,809
2,00000007,01,2,08,02,NaN,1,1,03,NaN,NaN,NaN,NaN,02,NaN,1,NaN,3,2,NaN,NaN,1,1,2,2,2,2,2,2,1,...,2,2,40,7,1,2,1,2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN,22,03,2,02,3,2,722
3,00000007,02,1,26,07,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,NaN,04,2,1,23,03,1,3,1,1,40,8,22,NaN,NaN,NaN,NaN,4,722
4,00000009,01,1,27,01,NaN,1,2,09,NaN,NaN,02,NaN,NaN,NaN,1,NaN,3,2,2,2,1,1,2,2,1,1,2,2,1,...,NaN,NaN,NaN,NaN,2,2,2,2,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,25,8,26,09,1,08,2,5,425


## 2. Diagnóstico de calidad de la base cruda

Antes de tocar nada se revisa: (a) duplicados, (b) que las llaves de unión sean únicas, (c) que todas las personas tengan vivienda, y (d) el patrón de valores faltantes.

In [3]:
diag = {
    "Filas duplicadas en TMODULO": int(mod.duplicated().sum()),
    "Filas duplicadas en TVIVIENDA": int(viv.duplicated().sum()),
    "Llave persona (FOLIO + N_REN) repetida": int(mod.duplicated(["FOLIO", "N_REN"]).sum()),
    "Llave vivienda (FOLIO) repetida": int(viv["FOLIO"].duplicated().sum()),
    "Personas sin vivienda asociada": int((~mod["FOLIO"].isin(viv["FOLIO"])).sum()),
    "Viviendas con personas de 0-29 años (P1_3 = 1)": int((viv["P1_3"] == "1").sum()),
    "Hogares distintos en TMODULO": int(mod["FOLIO"].nunique()),
}
pd.Series(diag, name="valor").to_frame()

,valor
Filas duplicadas en TMODULO,0
Filas duplicadas en TVIVIENDA,0
Llave persona (FOLIO + N_REN) repetida,0
Llave vivienda (FOLIO) repetida,0
Personas sin vivienda asociada,0
Viviendas con personas de 0-29 años (P1_3 = 1),14456
Hogares distintos en TMODULO,14456


**Resultado:** no hay duplicados ni llaves repetidas, y las 14,456 viviendas que declararon tener personas de 0 a 29 años son exactamente los 14,456 hogares de `TMODULO`. La unión 1 a muchos (vivienda → personas) es segura.

Sobre los faltantes: en esta encuesta casi todos los vacíos son **estructurales** (saltos del cuestionario). Por ejemplo, la razón de no inscripción (`PB3_2`) solo se pregunta a quien *no* se inscribió. Por eso **no se imputan ni se eliminan filas por tener vacíos**: se verifica que cada vacío corresponda a un salto válido (ver sección 7).

In [4]:
faltantes = (mod.isna().mean() * 100).round(1).sort_values(ascending=False)
faltantes.head(15).to_frame("% vacío (TMODULO)")

,% vacío (TMODULO)
PA3_3_BIMESTRE,100.0
PB3_5_TRIMESTRE,99.9
PB3_5_BIMESTRE,99.9
PA3_3_TRIMESTRE,99.9
PB3_5_MODMAT,99.9
PA3_3_MODMAT,99.6
PA3_3_CUATRIMESTRE,99.0
PB3_5_CUATRIMESTRE,99.0
PB3_4,99.0
PA3_5,98.8


## 3. Selección de variables

De las 118 variables se conservan solo las que responden la pregunta. Cada una se justifica abajo.

| Variable | Tabla | Pregunta del cuestionario | Para qué se usa |
|---|---|---|---|
| `FOLIO`, `N_REN` | ambas | Identificadores | Llave de unión y de persona |
| `ENT` | vivienda | Entidad federativa | Factor **territorial** (estado y región) |
| `FACTOR` | módulo | Factor de expansión de la persona | Estimaciones representativas |
| `SEXO`, `EDAD` | módulo | 2.2 y 2.3 | Segmentación por sexo y edad |
| `P3_1` | módulo | 3.1 Parentesco del informante con la persona | Estructura familiar (¿responde la madre/padre?) |
| `PA3_1`, `PA3_2`, `NIVEL_A` | módulo | A3.1–A3.3 Inscripción, tipo de escuela y nivel 2020-21 | Población en riesgo de abandono |
| `PA3_4`, `PA3_5` | módulo | A3.4–A3.5 ¿Concluyó 2020-21? y razón | Abandono **dentro** del ciclo |
| `PB3_1`, `PB3_2` | módulo | B3.1–B3.2 Inscripción 2021-22 y razón de no inscribirse | **Abandono reciente** y sus razones |
| `PC3_1`, `PC3_3_1`, `PC3_5`, `PC3_6`, `PC3_7` | módulo | C3.1–C3.7 Asistió alguna vez, nivel alcanzado, edad y razón por la que dejó la escuela, ¿piensa retomar? | **Abandono histórico** y sus razones |
| `PD3_1` | módulo | D3.1 Condición de actividad | Relación trabajo–abandono |
| `ESC` | módulo | Escolaridad (derivada por INEGI) | Logro educativo |
| `P1_1` | vivienda | 1.1 Número de residentes | Tamaño del hogar |
| `P1_4_1` … `P1_4_6` | vivienda | 1.4 Equipamiento TIC del hogar | **Proxy de ingreso / nivel socioeconómico** |
| `P1_5` | vivienda | 1.5 Razón de no tener internet | Restricción económica y **proxy de zona sin cobertura** |

> **Limitación central (se reporta en la presentación):** la ENAPE 2021 **no capta ingreso del hogar, escolaridad de los padres ni tamaño de localidad urbano/rural**. Para no dejar sin respuesta esos tres factores se construyen *proxies* documentados: equipamiento tecnológico del hogar (ingreso), si el informante es la madre/el padre (entorno familiar) y región/entidad + falta de cobertura de internet (zona).

In [5]:
VARS_VIV = ["FOLIO", "ENT", "P1_1"] + [f"P1_4_{i}" for i in range(1, 7)] + ["P1_5"]
VARS_MOD = ["FOLIO", "N_REN", "SEXO", "EDAD", "P3_1",
            "PA3_1", "PA3_2", "NIVEL_A", "PA3_4", "PA3_5",
            "PB3_1", "PB3_2",
            "PC3_1", "PC3_3_1", "PC3_5", "PC3_6", "PC3_7",
            "PD3_1", "ESC", "FACTOR"]

df = mod[VARS_MOD].merge(viv[VARS_VIV], on="FOLIO", how="left", validate="many_to_one")
print("Base unida:", df.shape)
df.head()

Base unida: (32343, 29)


,FOLIO,N_REN,SEXO,EDAD,P3_1,PA3_1,PA3_2,NIVEL_A,PA3_4,PA3_5,PB3_1,PB3_2,PC3_1,PC3_3_1,PC3_5,PC3_6,PC3_7,PD3_1,ESC,FACTOR,ENT,P1_1,P1_4_1,P1_4_2,P1_4_3,P1_4_4,P1_4_5,P1_4_6,P1_5
0,00000002,01,1,13,02,1,1,03,1,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,809,24,04,1,1,1,1,1,1,NaN
1,00000002,02,2,19,02,1,1,09,1,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,7,5,809,24,04,1,1,1,1,1,1,NaN
2,00000007,01,2,08,02,1,1,03,1,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,722,22,07,2,2,1,2,1,1,NaN
3,00000007,02,1,26,07,2,NaN,NaN,NaN,NaN,2,03,1,04,23,03,1,1,4,722,22,07,2,2,1,2,1,1,NaN
4,00000009,01,1,27,01,1,2,09,1,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,1,5,425,26,05,1,1,1,1,1,1,NaN


## 4. Cómo están clasificadas las variables

Los CSV de INEGI **no traen catálogos**: todo viene como códigos. Los catálogos siguientes se transcribieron del cuestionario oficial ENAPE 2021.

**Hallazgo de calidad:** en dos variables la base tiene un código más que el cuestionario publicado (`PB3_2` llega a 16 y `PC3_6` a 14). Se identificaron empíricamente en la celda siguiente:
- `PB3_2 = 16`: edad promedio ≈ 3 años → corresponde a *"aún está muy pequeño(a)"* (niñas y niños de 3-5 años).
- `PC3_6 = 14`: 91 % de esas personas llegó a licenciatura → corresponde a *"terminó sus estudios / logró su meta"*.
El resto de los códigos coincide con el cuestionario (p. ej. "se casó o embarazo" es 78 % mujeres y "quehaceres del hogar/cuidados" es 96-97 % mujeres, como se esperaría).

In [6]:
chk = df.assign(EDAD_N=df["EDAD"].astype(int))
evid_pb = chk[chk["PB3_2"].notna()].groupby("PB3_2").agg(
    casos=("EDAD_N", "size"), edad_media=("EDAD_N", "mean"),
    pct_mujeres=("SEXO", lambda s: (s == "2").mean() * 100))
evid_pc = chk[chk["PC3_6"].notna()].groupby("PC3_6").agg(
    casos=("EDAD_N", "size"), edad_media=("EDAD_N", "mean"),
    pct_licenciatura=("PC3_3_1", lambda s: (s == "08").mean() * 100))
display(evid_pb.round(1).tail(6))
display(evid_pc.round(1).tail(6))

,casos,edad_media,pct_mujeres
PB3_2,,,
11,703,23.4,77.5
12,297,24.8,96.6
13,43,14.6,65.1
14,158,18.0,43.7
15,288,17.0,49.7
16,305,3.1,48.9


,casos,edad_media,pct_licenciatura
PC3_6,,,
09,786,24.0,12.1
10,93,24.2,12.9
11,32,23.8,12.5
12,125,21.5,8.0
13,153,20.5,21.6
14,1625,26.2,91.3


In [7]:
ENTIDADES = {"01": "Aguascalientes", "02": "Baja California", "03": "Baja California Sur", "04": "Campeche",
    "05": "Coahuila", "06": "Colima", "07": "Chiapas", "08": "Chihuahua", "09": "Ciudad de México",
    "10": "Durango", "11": "Guanajuato", "12": "Guerrero", "13": "Hidalgo", "14": "Jalisco",
    "15": "México", "16": "Michoacán", "17": "Morelos", "18": "Nayarit", "19": "Nuevo León",
    "20": "Oaxaca", "21": "Puebla", "22": "Querétaro", "23": "Quintana Roo", "24": "San Luis Potosí",
    "25": "Sinaloa", "26": "Sonora", "27": "Tabasco", "28": "Tamaulipas", "29": "Tlaxcala",
    "30": "Veracruz", "31": "Yucatán", "32": "Zacatecas"}
REGION = {**{k: "Norte" for k in ["02", "03", "05", "08", "10", "19", "25", "26", "28"]},
          **{k: "Centro-Occidente" for k in ["01", "06", "11", "14", "16", "18", "22", "24", "32"]},
          **{k: "Centro" for k in ["09", "13", "15", "17", "21", "29"]},
          **{k: "Sur-Sureste" for k in ["04", "07", "12", "20", "23", "27", "30", "31"]}}

CAT = {
    "SEXO": {"1": "Hombre", "2": "Mujer"},
    "SI_NO": {"1": "Sí", "2": "No"},
    "TIPO_ESC": {"1": "Pública", "2": "Privada"},
    "P3_1": {"01": "Madre", "02": "Padre", "03": "Cónyuge", "04": "Tío(a)", "05": "Primo(a)",
             "06": "Abuelo(a)", "07": "Hermano(a)", "08": "Otro pariente", "09": "Sin parentesco",
             "10": "La propia persona"},
    "NIVEL_A": {"01": "Educación inicial", "02": "Preescolar", "03": "Primaria", "04": "Secundaria",
                "05": "Profesional técnico", "06": "Bachillerato general", "07": "Bachillerato tecnológico",
                "08": "Técnico superior universitario", "09": "Licenciatura", "10": "Especialidad",
                "11": "Maestría", "12": "Doctorado"},
    "PC3_3_1": {"00": "Ninguno", "01": "Preescolar", "02": "Primaria", "03": "Secundaria",
                "04": "Profesional técnico", "05": "Bachillerato general", "06": "Bachillerato tecnológico",
                "07": "Técnico superior universitario", "08": "Licenciatura", "09": "Especialidad",
                "10": "Maestría", "11": "Doctorado"},
    "ESC": {"0": "Ninguna", "1": "Preescolar", "2": "Primaria", "3": "Secundaria",
            "4": "Media superior", "5": "Superior", "6": "Posgrado"},
    "PA3_5": {"01": "Falta de dinero o recursos", "02": "Tenía que trabajar", "03": "Falta de interés o aptitud",
              "04": "Reprobó materias / bajo rendimiento", "05": "Escuela cerrada o lejana", "06": "Discapacidad",
              "07": "Pandemia COVID-19", "08": "Se enfermó de COVID-19", "09": "Problemas de salud",
              "10": "Se unió, casó o embarazo", "11": "Quehaceres del hogar o cuidados",
              "12": "La familia no lo permitió", "13": "Problemas personales", "14": "Otra razón"},
    "PB3_2": {"01": "Falta de dinero o recursos", "02": "Tenía que trabajar", "03": "Falta de interés o aptitud",
              "04": "Logró su meta educativa", "05": "Reprobó materias / bajo rendimiento",
              "06": "No hay escuela cercana", "07": "Discapacidad", "08": "Pandemia COVID-19",
              "09": "Se enfermó de COVID-19", "10": "Problemas de salud", "11": "Se unió, casó o embarazo",
              "12": "Quehaceres del hogar o cuidados", "13": "La familia no lo permitió",
              "14": "Problemas personales", "15": "Otra razón", "16": "Aún está muy pequeño(a)"},
    "PC3_6": {"01": "Falta de dinero o recursos", "02": "Tenía que trabajar", "03": "Falta de interés o aptitud",
              "04": "Reprobó materias / bajo rendimiento", "05": "No hay escuela cercana",
              "06": "No cumplió requisitos de ingreso", "07": "Discapacidad", "08": "Problemas de salud",
              "09": "Se unió, casó o embarazo", "10": "Quehaceres del hogar o cuidados",
              "11": "La familia no lo permitió", "12": "Problemas personales", "13": "Otra razón",
              "14": "Terminó sus estudios / logró su meta"},
    "PC3_7": {"1": "Sí", "2": "No", "9": "No sabe"},
    "PD3_1": {"1": "Trabajó", "2": "Vendió o elaboró productos", "3": "Ayudó en negocio familiar o campo",
              "4": "Trabajo no de negocio (lavar, cuidar niños…)", "5": "Aprendiz o servicio social",
              "6": "Buscó trabajo", "7": "Estudia u otra situación"},
    "P1_5": {"01": "Falta de dinero", "02": "No le interesa / no lo necesita", "03": "No sabe usarlo",
             "04": "Desconoce su utilidad", "05": "Equipo insuficiente", "06": "No hay servicio en la zona",
             "07": "Tiene acceso en otro lugar", "08": "Privacidad o seguridad", "09": "Otra razón",
             "99": "No responde"},
}
TIC = {"P1_4_1": "computadora de escritorio", "P1_4_2": "laptop", "P1_4_3": "televisión digital",
       "P1_4_4": "tablet", "P1_4_5": "smartphone", "P1_4_6": "internet fijo"}

# Verificación: todo código observado en la base existe en su catálogo
mapa_var_cat = {"SEXO": "SEXO", "P3_1": "P3_1", "NIVEL_A": "NIVEL_A", "PC3_3_1": "PC3_3_1", "ESC": "ESC",
                "PA3_5": "PA3_5", "PB3_2": "PB3_2", "PC3_6": "PC3_6", "PC3_7": "PC3_7", "PD3_1": "PD3_1",
                "P1_5": "P1_5", "PA3_1": "SI_NO", "PA3_4": "SI_NO", "PB3_1": "SI_NO", "PC3_1": "SI_NO",
                "PA3_2": "TIPO_ESC", **{k: "SI_NO" for k in TIC}}
sin_catalogo = {v: sorted(set(df[v].dropna()) - set(CAT[c])) for v, c in mapa_var_cat.items()}
sin_catalogo = {k: v for k, v in sin_catalogo.items() if v}
print("Códigos sin catálogo:", sin_catalogo if sin_catalogo else "ninguno ✔")
assert set(df["ENT"]) <= set(ENTIDADES)

Códigos sin catálogo: ninguno ✔


## 5. Limpieza con **Data Wrangler** (extensión de VS Code / GitHub Codespaces)

**Cómo se hizo en Codespaces:**
1. Instalar la extensión *Data Wrangler* (Microsoft) desde el panel de extensiones.
2. Ejecutar la celda de arriba para tener `df` en memoria y pulsar **"Open 'df' in Data Wrangler"** (aparece bajo la salida del DataFrame, o en *Jupyter: Variables* → ícono de Data Wrangler).
3. En Data Wrangler revisar el panel **Summary** de cada columna (tipo, % faltantes, valores únicos, distribución) y aplicar las operaciones: *Change column type*, *Find and replace*, *Rename column*, *New column by formula*, *Drop column*.
4. Pulsar **"Export to notebook"**: Data Wrangler genera la función `clean_data(df)` de abajo (mismo formato: un comentario por paso).

Decisiones de limpieza y su justificación:
- **Tipos:** `EDAD`, `P1_1`, `PC3_5` a entero y `FACTOR` a flotante; códigos categóricos se quedan como texto y se crea su versión con etiqueta.
- **`PC3_5 = 98` ("no recuerda")** se convierte a vacío: es un código de no respuesta, no una edad (dejarlo inflaría la edad media de abandono).
- **`PC3_7 = 9` ("no sabe")** se mantiene como categoría propia: es información, no un error.
- **Equipamiento TIC** se recodifica Sí=1 / No=0 para poder sumarlo en un índice.
- **Nombres:** se renombran a nombres descriptivos en minúsculas (`snake_case`).

In [8]:
def clean_data(df):
    # Change column type to int64 for columns: 'EDAD', 'P1_1'
    df = df.astype({"EDAD": "int64", "P1_1": "int64"})
    # Change column type to float64 for column: 'FACTOR'
    df = df.astype({"FACTOR": "float64"})
    # Replace "98" (no recuerda) with missing value in column: 'PC3_5'
    df["PC3_5"] = pd.to_numeric(df["PC3_5"].replace("98", np.nan)).astype("Int64")
    # Replace "1"/"2" with 1/0 in TIC columns: 'P1_4_1' ... 'P1_4_6'
    for c in TIC:
        df[c] = df[c].map({"1": 1, "2": 0}).astype("Int64")
    # New columns with category labels (find & replace using official catalogs)
    df["sexo"] = df["SEXO"].map(CAT["SEXO"])
    df["entidad"] = df["ENT"].map(ENTIDADES)
    df["region"] = df["ENT"].map(REGION)
    df["informante_parentesco"] = df["P3_1"].map(CAT["P3_1"])
    df["inscrito_2020_21"] = df["PA3_1"].map(CAT["SI_NO"])
    df["tipo_escuela_2020_21"] = df["PA3_2"].map(CAT["TIPO_ESC"])
    df["nivel_2020_21"] = df["NIVEL_A"].map(CAT["NIVEL_A"])
    df["concluyo_2020_21"] = df["PA3_4"].map(CAT["SI_NO"])
    df["razon_no_concluyo_2020_21"] = df["PA3_5"].map(CAT["PA3_5"])
    df["inscrito_2021_22"] = df["PB3_1"].map(CAT["SI_NO"])
    df["razon_no_inscripcion_2021_22"] = df["PB3_2"].map(CAT["PB3_2"])
    df["asistio_alguna_vez"] = df["PC3_1"].map(CAT["SI_NO"])
    df["nivel_alcanzado"] = df["PC3_3_1"].map(CAT["PC3_3_1"])
    df["razon_dejo_escuela"] = df["PC3_6"].map(CAT["PC3_6"])
    df["piensa_retomar"] = df["PC3_7"].map(CAT["PC3_7"])
    df["condicion_actividad"] = df["PD3_1"].map(CAT["PD3_1"])
    df["escolaridad"] = df["ESC"].map(CAT["ESC"])
    df["razon_sin_internet"] = df["P1_5"].map(CAT["P1_5"])
    # Rename columns
    df = df.rename(columns={"FOLIO": "folio", "N_REN": "renglon", "EDAD": "edad", "FACTOR": "factor",
                            "P1_1": "tam_hogar", "PC3_5": "edad_dejo_escuela", **{k: f"tic_{v.split()[0]}"
                            if k != "P1_4_6" else "tic_internet" for k, v in TIC.items()}})
    df = df.rename(columns={"tic_computadora": "tic_computadora", "tic_televisión": "tic_tv"})
    return df

df_clean = clean_data(df.copy())
df_clean.head()

,folio,renglon,SEXO,edad,P3_1,PA3_1,PA3_2,NIVEL_A,PA3_4,PA3_5,PB3_1,PB3_2,PC3_1,PC3_3_1,edad_dejo_escuela,PC3_6,PC3_7,PD3_1,ESC,factor,ENT,tam_hogar,tic_computadora,tic_laptop,tic_tv,tic_tablet,tic_smartphone,tic_internet,P1_5,sexo,entidad,region,informante_parentesco,inscrito_2020_21,tipo_escuela_2020_21,nivel_2020_21,concluyo_2020_21,razon_no_concluyo_2020_21,inscrito_2021_22,razon_no_inscripcion_2021_22,asistio_alguna_vez,nivel_alcanzado,razon_dejo_escuela,piensa_retomar,condicion_actividad,escolaridad,razon_sin_internet
0,00000002,01,1,13,02,1,1,03,1,NaN,1,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,2,809.0,24,4,1,1,1,1,1,1,NaN,Hombre,San Luis Potosí,Centro-Occidente,Padre,Sí,Pública,Primaria,Sí,NaN,Sí,NaN,NaN,NaN,NaN,NaN,NaN,Primaria,NaN
1,00000002,02,2,19,02,1,1,09,1,NaN,1,NaN,NaN,NaN,<NA>,NaN,NaN,7,5,809.0,24,4,1,1,1,1,1,1,NaN,Mujer,San Luis Potosí,Centro-Occidente,Padre,Sí,Pública,Licenciatura,Sí,NaN,Sí,NaN,NaN,NaN,NaN,NaN,Estudia u otra situación,Superior,NaN
2,00000007,01,2,8,02,1,1,03,1,NaN,1,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,2,722.0,22,7,0,0,1,0,1,1,NaN,Mujer,Querétaro,Centro-Occidente,Padre,Sí,Pública,Primaria,Sí,NaN,Sí,NaN,NaN,NaN,NaN,NaN,NaN,Primaria,NaN
3,00000007,02,1,26,07,2,NaN,NaN,NaN,NaN,2,03,1,04,23,03,1,1,4,722.0,22,7,0,0,1,0,1,1,NaN,Hombre,Querétaro,Centro-Occidente,Hermano(a),No,NaN,NaN,NaN,NaN,No,Falta de interés o aptitud,Sí,Profesional técnico,Falta de interés o aptitud,Sí,Trabajó,Media superior,NaN
4,00000009,01,1,27,01,1,2,09,1,NaN,1,NaN,NaN,NaN,<NA>,NaN,NaN,1,5,425.0,26,5,1,1,1,1,1,1,NaN,Hombre,Sonora,Norte,Madre,Sí,Privada,Licenciatura,Sí,NaN,Sí,NaN,NaN,NaN,NaN,NaN,Trabajó,Superior,NaN


## 6. Variables derivadas (definición operativa)

**a) Abandono escolar reciente** (`abandono_reciente`, variable dependiente principal). Población: personas de **6 a 29 años inscritas en 2020-2021**.
- `1` = no se inscribió en 2021-2022 **por cualquier razón distinta de "logró su meta educativa"**.
- `0` = sí se inscribió en 2021-2022.
- vacío = quienes no estaban inscritos en 2020-2021 o salieron porque terminaron lo que buscaban (egresar no es abandonar).

Se excluyen 3-5 años porque en preescolar la no inscripción responde sobre todo a la edad ("aún está muy pequeño"), no a abandono.

**b) Abandono histórico** (`abandono_historico`). Personas de 6-29 años que asistieron alguna vez, no están inscritas en ningún ciclo y **no** dejaron la escuela por haber terminado (`PC3_6 ≠ 14`). Complementa a (a): muestra las razones estructurales, más allá del choque de la pandemia.

**c) Proxy de ingreso / nivel socioeconómico** (`indice_tic`, `nse_tic`). La ENAPE no pregunta ingreso, así que se suma el equipamiento del hogar (computadora de escritorio, laptop, TV digital, tablet, smartphone, internet fijo), de 0 a 6 bienes. Los activos del hogar son un proxy estándar de bienestar cuando falta el ingreso. Cortes por distribución ponderada: **Bajo** 0-2 (≈ 33 % de la población), **Medio** 3-4 (≈ 47 %), **Alto** 5-6 (≈ 20 %).

**d) Proxy de zona** (`sin_cobertura_internet`): hogar sin internet porque *no hay servicio en la zona* (señal de localidad remota o rural), y la **región** del país.

**e) Entorno familiar** (`informante_padre_madre`): el informante es la madre o el padre. Es el dato de la encuesta más cercano a los padres; **no** sustituye su escolaridad.

**f) Otras:** grupo de edad, nivel agrupado 2020-21, trabaja (`PD3_1` en 1-5), hogar con internet, tamaño del hogar agrupado.

In [9]:
def derivar(d):
    d = d.copy()
    d["grupo_edad"] = pd.cut(d["edad"], [-1, 2, 5, 11, 14, 17, 22, 29],
        labels=["0-2", "3-5", "6-11 (primaria)", "12-14 (secundaria)", "15-17 (media sup.)",
                "18-22 (superior)", "23-29"])
    elegibles = (d["edad"] >= 6) & (d["PA3_1"] == "1")
    sale = (d["PB3_1"] == "2") & (d["PB3_2"] != "04")
    sigue = d["PB3_1"] == "1"
    d["abandono_reciente"] = np.select([elegibles & sale, elegibles & sigue], [1, 0], default=-1)
    d["abandono_reciente"] = d["abandono_reciente"].replace(-1, np.nan)

    d["abandono_historico"] = np.where(
        (d["edad"] >= 6) & (d["PC3_1"] == "1") & d["PC3_6"].notna(),
        (d["PC3_6"] != "14").astype(int), np.nan)

    tic_cols = ["tic_computadora", "tic_laptop", "tic_tv", "tic_tablet", "tic_smartphone", "tic_internet"]
    d["indice_tic"] = d[tic_cols].sum(axis=1).astype(int)
    d["nse_tic"] = pd.cut(d["indice_tic"], [-1, 2, 4, 6], labels=["Bajo (0-2 bienes)", "Medio (3-4)", "Alto (5-6)"])
    d["sin_internet_por_dinero"] = (d["P1_5"] == "01").astype(int)
    d["sin_cobertura_internet"] = (d["P1_5"] == "06").astype(int)
    d["informante_padre_madre"] = d["P3_1"].isin(["01", "02"]).map({True: "Madre/padre", False: "Otro familiar o la persona"})
    d["tam_hogar_grupo"] = pd.cut(d["tam_hogar"], [0, 3, 5, 99], labels=["1-3 personas", "4-5 personas", "6 o más"])
    d["nivel_agrupado_2020_21"] = d["NIVEL_A"].map({"01": "Preescolar/inicial", "02": "Preescolar/inicial",
        "03": "Primaria", "04": "Secundaria", "05": "Media superior", "06": "Media superior",
        "07": "Media superior", **{k: "Superior" for k in ["08", "09", "10", "11", "12"]}})
    d["trabaja"] = np.where(d["PD3_1"].isna(), np.nan, d["PD3_1"].isin(list("12345")).astype(int))
    return d

df_final = derivar(df_clean)
df_final[["edad", "sexo", "inscrito_2020_21", "inscrito_2021_22", "razon_no_inscripcion_2021_22",
          "abandono_reciente", "indice_tic", "nse_tic", "region"]].sample(8, random_state=1)

,edad,sexo,inscrito_2020_21,inscrito_2021_22,razon_no_inscripcion_2021_22,abandono_reciente,indice_tic,nse_tic,region
11895,2,Mujer,NaN,NaN,NaN,NaN,1,Bajo (0-2 bienes),Centro
29984,2,Hombre,NaN,NaN,NaN,NaN,4,Medio (3-4),Centro-Occidente
15594,0,Hombre,NaN,NaN,NaN,NaN,5,Alto (5-6),Sur-Sureste
20407,6,Hombre,No,No,No hay escuela cercana,NaN,2,Bajo (0-2 bienes),Norte
13380,21,Hombre,Sí,No,Otra razón,1.0,2,Bajo (0-2 bienes),Sur-Sureste
27544,13,Hombre,Sí,Sí,NaN,0.0,4,Medio (3-4),Norte
11227,14,Mujer,Sí,Sí,NaN,0.0,4,Medio (3-4),Centro-Occidente
26747,20,Hombre,No,No,Tenía que trabajar,NaN,1,Bajo (0-2 bienes),Sur-Sureste


## 7. Validaciones de consistencia

Cada prueba confirma que la base limpia es coherente con el cuestionario y con las cifras oficiales de INEGI. Si alguna falla, el notebook se detiene (`assert`).

In [10]:
w = df_final["factor"]
val = []
def prueba(nombre, ok, detalle=""):
    val.append({"prueba": nombre, "resultado": "✔" if ok else "✘", "detalle": detalle})
    assert ok, nombre

prueba("Edad entre 0 y 29", df_final["edad"].between(0, 29).all())
prueba("Factor de expansión positivo y sin vacíos", (w > 0).all() and w.notna().all())
prueba("Población 0-29 expandida ≈ 60.0 M (INEGI)", abs(w.sum() / 1e6 - 60.0) < 0.1, f"{w.sum()/1e6:.2f} millones")
pct_h = w[df_final["sexo"] == "Hombre"].sum() / w.sum() * 100
prueba("Hombres ≈ 50.2 % (INEGI)", abs(pct_h - 50.2) < 0.2, f"{pct_h:.1f} %")
no_insc = w[(df_final["edad"] >= 3) & (df_final["PB3_1"] == "2")].sum() / 1e6
prueba("No inscritos 2021-22 (3-29 años) ≈ 22.3 M (INEGI)", abs(no_insc - 22.3) < 0.15, f"{no_insc:.2f} millones")
ret = df_final[df_final["PC3_7"].notna()]
pr = lambda s: (ret.loc[s, "factor"] * (ret.loc[s, "PC3_7"] == "1")).sum() / ret.loc[s, "factor"].sum() * 100
prueba("Piensa retomar estudios ≈ 28.6 % (INEGI)", abs(pr(ret.index) - 28.6) < 0.3, f"{pr(ret.index):.1f} %")
prueba("Razón de no inscripción solo si no se inscribió",
       (df_final["PB3_2"].notna() == (df_final["PB3_1"] == "2")).all())
prueba("Razón de no conclusión solo si no concluyó",
       (df_final["PA3_5"].notna() == (df_final["PA3_4"] == "2")).all())
prueba("0-2 años sin preguntas de inscripción", df_final.loc[df_final["edad"] <= 2, "PA3_1"].isna().all())
prueba("Edad a la que dejó la escuela ≤ edad actual",
       (df_final["edad_dejo_escuela"].dropna() <= df_final.loc[df_final["edad_dejo_escuela"].notna(), "edad"]).all())
prueba("Índice TIC sin vacíos y entre 0 y 6", df_final["indice_tic"].between(0, 6).all())
prueba("Abandono reciente solo en 6-29 inscritos 2020-21",
       df_final.loc[df_final["abandono_reciente"].notna(), "edad"].min() >= 6)
pd.DataFrame(val)

,prueba,resultado,detalle
0,Edad entre 0 y 29,✔,
1,Factor de expansión positivo y sin vacíos,✔,
2,Población 0-29 expandida ≈ 60.0 M (INEGI),✔,59.98 millones
3,Hombres ≈ 50.2 % (INEGI),✔,50.2 %
4,No inscritos 2021-22 (3-29 años) ≈ 22.3 M (INEGI),✔,22.32 millones
5,Piensa retomar estudios ≈ 28.6 % (INEGI),✔,28.6 %
6,Razón de no inscripción solo si no se inscribió,✔,
7,Razón de no conclusión solo si no concluyó,✔,
8,0-2 años sin preguntas de inscripción,✔,
9,Edad a la que dejó la escuela ≤ edad actual,✔,


Las cifras expandidas coinciden con la nota técnica de INEGI (60.0 M de personas de 0-29 años; 50.2 % hombres; 22.3 M no inscritos en 2021-22; 28.6 % piensa retomar), lo que confirma que la lectura, la unión de tablas y el uso del factor de expansión son correctos.

## 8. Limitaciones de calidad identificadas
1. **Encuesta telefónica:** representa al 96.6 % de la población con teléfono; los hogares más pobres y aislados (sin teléfono) están subrepresentados, lo que probablemente **subestima** el abandono en el nivel socioeconómico bajo.
2. **Sin ingreso, escolaridad de los padres ni tamaño de localidad:** se usan proxies (equipamiento TIC, parentesco del informante, región y cobertura de internet). Las conclusiones sobre esos factores son de asociación con el proxy, no con la variable original.
3. **Informante indirecto:** un integrante del hogar responde por los demás; las razones de abandono son las que percibe el informante.
4. **Contexto de pandemia:** el ciclo 2021-22 inició en transición al regreso presencial; el abandono reciente mezcla un choque coyuntural con factores estructurales (por eso se analiza también el abandono histórico).
5. **Códigos no documentados** (`PB3_2 = 16`, `PC3_6 = 14`) se etiquetaron por evidencia empírica (sección 4).
6. **`PC3_5 = 98`** (no recuerda) se trató como faltante: afecta a pocos casos.

## 9. Exportación de la base final

Se exporta **solo con las variables empleadas** en el análisis: CSV (para el notebook de análisis) y Excel con 5 hojas: base limpia, diccionario de variables, catálogos, bitácora de limpieza y validaciones.

In [11]:
COLS_FINALES = ["folio", "renglon", "factor", "entidad", "region", "sexo", "edad", "grupo_edad",
    "informante_parentesco", "informante_padre_madre", "tam_hogar", "tam_hogar_grupo",
    "inscrito_2020_21", "tipo_escuela_2020_21", "nivel_2020_21", "nivel_agrupado_2020_21",
    "concluyo_2020_21", "razon_no_concluyo_2020_21", "inscrito_2021_22", "razon_no_inscripcion_2021_22",
    "abandono_reciente", "asistio_alguna_vez", "nivel_alcanzado", "edad_dejo_escuela", "razon_dejo_escuela",
    "abandono_historico", "piensa_retomar", "condicion_actividad", "trabaja", "escolaridad",
    "tic_computadora", "tic_laptop", "tic_tv", "tic_tablet", "tic_smartphone", "tic_internet",
    "indice_tic", "nse_tic", "razon_sin_internet", "sin_internet_por_dinero", "sin_cobertura_internet"]
base = df_final[COLS_FINALES].copy()

DESC = {"folio": ("Identificador de la vivienda", "FOLIO"), "renglon": ("Número de persona en el hogar", "N_REN"),
    "factor": ("Factor de expansión de la persona", "FACTOR"), "entidad": ("Entidad federativa", "ENT"),
    "region": ("Región (agrupación de entidades)", "derivada de ENT"), "sexo": ("Sexo", "SEXO"),
    "edad": ("Edad en años cumplidos", "EDAD"), "grupo_edad": ("Grupo de edad por etapa escolar", "derivada de EDAD"),
    "informante_parentesco": ("Parentesco del informante con la persona", "P3_1"),
    "informante_padre_madre": ("El informante es la madre o el padre", "derivada de P3_1"),
    "tam_hogar": ("Número de residentes del hogar", "P1_1"), "tam_hogar_grupo": ("Tamaño del hogar agrupado", "derivada de P1_1"),
    "inscrito_2020_21": ("Estuvo inscrito en el ciclo 2020-2021", "PA3_1"),
    "tipo_escuela_2020_21": ("Escuela pública o privada 2020-2021", "PA3_2"),
    "nivel_2020_21": ("Nivel cursado en 2020-2021", "NIVEL_A"),
    "nivel_agrupado_2020_21": ("Nivel 2020-2021 agrupado", "derivada de NIVEL_A"),
    "concluyo_2020_21": ("Concluyó el ciclo 2020-2021", "PA3_4"),
    "razon_no_concluyo_2020_21": ("Razón principal de no concluir 2020-2021", "PA3_5"),
    "inscrito_2021_22": ("Está inscrito en el ciclo 2021-2022", "PB3_1"),
    "razon_no_inscripcion_2021_22": ("Razón principal de no inscribirse en 2021-2022", "PB3_2"),
    "abandono_reciente": ("1 = inscrito 2020-21 que no se inscribió 2021-22 (excluye 'logró meta'); 0 = continuó; 6-29 años", "derivada de PA3_1, PB3_1, PB3_2"),
    "asistio_alguna_vez": ("Asistió alguna vez a la escuela (no inscritos en ningún ciclo)", "PC3_1"),
    "nivel_alcanzado": ("Último nivel alcanzado (no inscritos)", "PC3_3_1"),
    "edad_dejo_escuela": ("Edad a la que dejó la escuela (98 → vacío)", "PC3_5"),
    "razon_dejo_escuela": ("Razón principal por la que dejó la escuela", "PC3_6"),
    "abandono_historico": ("1 = dejó la escuela sin terminar; 0 = dejó porque terminó; 6-29 años", "derivada de PC3_1, PC3_6"),
    "piensa_retomar": ("Piensa retomar sus estudios", "PC3_7"),
    "condicion_actividad": ("Actividad la semana anterior", "PD3_1"),
    "trabaja": ("1 = realizó alguna actividad económica (PD3_1 de 1 a 5)", "derivada de PD3_1"),
    "escolaridad": ("Escolaridad (nivel)", "ESC"),
    **{f"tic_{n}": (f"Hogar tiene {t} (1 sí / 0 no)", k) for k, n, t in [
        ("P1_4_1", "computadora", "computadora de escritorio"), ("P1_4_2", "laptop", "laptop"),
        ("P1_4_3", "tv", "televisión digital"), ("P1_4_4", "tablet", "tablet"),
        ("P1_4_5", "smartphone", "smartphone"), ("P1_4_6", "internet", "internet fijo")]},
    "indice_tic": ("Número de bienes TIC en el hogar (0-6): proxy de ingreso", "derivada de P1_4_1 a P1_4_6"),
    "nse_tic": ("Nivel socioeconómico proxy según índice TIC", "derivada de indice_tic"),
    "razon_sin_internet": ("Razón de no tener internet fijo", "P1_5"),
    "sin_internet_por_dinero": ("1 = sin internet por falta de dinero", "derivada de P1_5"),
    "sin_cobertura_internet": ("1 = sin internet porque no hay servicio en la zona (proxy de zona remota)", "derivada de P1_5")}
dicc = pd.DataFrame([{"variable": c, "descripcion": DESC[c][0], "origen_enape": DESC[c][1],
                      "tipo": str(base[c].dtype), "% vacío": round(base[c].isna().mean() * 100, 1),
                      "valores_unicos": base[c].nunique()} for c in COLS_FINALES])

catalogos = pd.concat([pd.DataFrame({"catalogo": k, "codigo": list(v), "etiqueta": list(v.values())})
                       for k, v in CAT.items()], ignore_index=True)
bitacora = pd.DataFrame({"paso": range(1, 11), "accion": [
    "Carga de TVIVIENDA y TMODULO como texto para conservar ceros iniciales",
    "Diagnóstico: 0 duplicados, llaves únicas, 100 % de personas con vivienda",
    "Selección de 30 variables relevantes de 118 originales",
    "Unión persona–vivienda por FOLIO (muchos a uno, validada)",
    "Conversión de tipos (edad, tamaño de hogar, factor)",
    "PC3_5 = 98 (no recuerda) → vacío",
    "TIC Sí/No → 1/0; etiquetas con catálogos oficiales; códigos no documentados identificados empíricamente",
    "Renombrado a snake_case",
    "Variables derivadas: abandono reciente e histórico, índice TIC/NSE, proxies de zona y familia",
    "Validación contra cifras oficiales INEGI (población, % hombres, no inscritos, intención de retomar)"]})

base.to_csv(LIMPIOS / "enape2021_abandono_limpia.csv", index=False, encoding="utf-8-sig")
with pd.ExcelWriter(LIMPIOS / "ENAPE2021_abandono_escolar.xlsx", engine="xlsxwriter") as xw:
    for nombre, tabla in [("base_limpia", base), ("diccionario", dicc), ("catalogos", catalogos),
                          ("bitacora_limpieza", bitacora), ("validaciones", pd.DataFrame(val))]:
        tabla.to_excel(xw, sheet_name=nombre, index=False)
        ws = xw.sheets[nombre]
        ws.freeze_panes(1, 0)
        ws.autofilter(0, 0, len(tabla), len(tabla.columns) - 1)
        hdr = xw.book.add_format({"bold": True, "bg_color": "#1F3A5F", "font_color": "white", "text_wrap": True})
        for i, c in enumerate(tabla.columns):
            ws.write(0, i, c, hdr)
            ancho = min(60, max(10, int(tabla[c].astype(str).str.len().quantile(0.9)) + 2, len(c) + 2))
            ws.set_column(i, i, ancho)
print("Base final:", base.shape)
print("Archivos:", *sorted(p.name for p in LIMPIOS.iterdir()), sep="\n  ")
dicc

Base final: (32343, 41)
Archivos:
  ENAPE2021_abandono_escolar.xlsx
  enape2021_abandono_limpia.csv


,variable,descripcion,origen_enape,tipo,% vacío,valores_unicos
0,folio,Identificador de la vivienda,FOLIO,str,0.0,14456
1,renglon,Número de persona en el hogar,N_REN,str,0.0,13
2,factor,Factor de expansión de la persona,FACTOR,float64,0.0,3881
3,entidad,Entidad federativa,ENT,str,0.0,32
4,region,Región (agrupación de entidades),derivada de ENT,str,0.0,4
5,sexo,Sexo,SEXO,str,0.0,2
6,edad,Edad en años cumplidos,EDAD,int64,0.0,30
7,grupo_edad,Grupo de edad por etapa escolar,derivada de EDAD,category,0.0,7
8,informante_parentesco,Parentesco del informante con la persona,P3_1,str,0.0,10
9,informante_padre_madre,El informante es la madre o el padre,derivada de P3_1,str,0.0,2
